# LoRA Adapter Swap Optimization — Results Analysis

This notebook loads the sweep CSV produced by `experiments/run_sweep.py` and generates:
1. Cost ratio (Gurobi vs GA) across problem sizes
2. Solver wall time vs instance size
3. Per-component cost breakdown (latency / swap / priority)

Run `python experiments/run_sweep.py` before opening this notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from pathlib import Path

SWEEP_CSV = Path("../results/sweep.csv")
OUT_DIR   = Path("../results/charts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SWEEP_CSV)
print(f"Loaded {len(df)} rows  |  solvers: {df['solver'].unique()}")
df.head()

## 1. Cost ratio: Gurobi vs Genetic Algorithm

For each (sweep_param, sweep_value, seed) cell we compute:

```
gap% = (GA_total - Gurobi_total) / Gurobi_total * 100
```

A positive value means GA is worse than Gurobi by that percentage.

In [ ]:
gurobi = df[df.solver == "gurobi"].set_index(["seed", "sweep_param", "sweep_value"])
ga     = df[df.solver == "genetic"].set_index(["seed", "sweep_param", "sweep_value"])

ratio = ga[["total"]].join(gurobi[["total"]], lsuffix="_ga", rsuffix="_gurobi")
ratio["gap_pct"] = (ratio["total_ga"] - ratio["total_gurobi"]) / ratio["total_gurobi"] * 100
ratio = ratio.reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, param in zip(axes, ["n_requests", "n_adapters"]):
    sub = ratio[ratio.sweep_param == param]
    agg = sub.groupby("sweep_value")["gap_pct"].agg(["mean", "std"]).reset_index()
    ax.bar(agg["sweep_value"].astype(str), agg["mean"], yerr=agg["std"],
           capsize=4, color="#4878CF", alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel(param)
    ax.set_ylabel("GA gap above Gurobi (%)")
    ax.set_title(f"Optimality gap vs {param}")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())

fig.suptitle("Genetic Algorithm vs Gurobi — Cost Gap (lower is better for GA)", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "gap_vs_size.png", dpi=150)
plt.show()
print(f"Mean gap across all cells: {ratio['gap_pct'].mean():.1f}%")

## 2. Solver wall time vs instance size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = {"gurobi": "#D65F5F", "genetic": "#6ACC65"}

for ax, param in zip(axes, ["n_requests", "n_adapters"]):
    for solver, grp in df.groupby("solver"):
        sub = grp[grp.sweep_param == param]
        agg = sub.groupby("sweep_value")["wall_time"].agg(["mean", "std"]).reset_index()
        ax.plot(agg["sweep_value"], agg["mean"], marker="o",
                label=solver, color=colors[solver])
        ax.fill_between(
            agg["sweep_value"],
            (agg["mean"] - agg["std"]).clip(0),
            agg["mean"] + agg["std"],
            alpha=0.2, color=colors[solver]
        )
    ax.set_xlabel(param)
    ax.set_ylabel("Wall time (s)")
    ax.set_title(f"Solve time vs {param}")
    ax.legend()

fig.suptitle("Solver Wall Time Comparison", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "wall_time_vs_size.png", dpi=150)
plt.show()

## 3. Per-component cost breakdown

Shows how much of the total cost comes from latency, swap overhead, and priority penalty for each solver across problem sizes.

In [ ]:
components = ["latency", "swap", "priority"]
comp_colors = {"latency": "#4878CF", "swap": "#6ACC65", "priority": "#D65F5F"}

for param in ["n_requests", "n_adapters"]:
    sub = df[df.sweep_param == param]
    agg = sub.groupby(["sweep_value", "solver"])[components].mean().reset_index()

    values = sorted(agg["sweep_value"].unique())
    solvers = ["gurobi", "genetic"]
    x = np.arange(len(values))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(values) * 2), 5))

    for s_idx, solver in enumerate(solvers):
        s_data = agg[agg.solver == solver].set_index("sweep_value").reindex(values)
        bottoms = np.zeros(len(values))
        for comp in components:
            vals = s_data[comp].fillna(0).values
            bars = ax.bar(
                x + s_idx * width, vals, width,
                bottom=bottoms,
                label=f"{solver}/{comp}" if s_idx == 0 else f"_{comp}",
                color=comp_colors[comp],
                alpha=0.85 if solver == "gurobi" else 0.5,
                hatch="" if solver == "gurobi" else "//",
            )
            bottoms += vals

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(values)
    ax.set_xlabel(param)
    ax.set_ylabel("Cost")
    ax.set_title(f"Cost breakdown by component — {param} sweep  (solid=Gurobi, hatch=GA)")

    # build a clean legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=comp_colors[c], label=c) for c in components]
    legend_elements += [
        Patch(facecolor="grey", alpha=0.85, label="Gurobi"),
        Patch(facecolor="grey", alpha=0.5, hatch="//", label="Genetic"),
    ]
    ax.legend(handles=legend_elements, fontsize=8)

    plt.tight_layout()
    plt.savefig(OUT_DIR / f"cost_components_{param}.png", dpi=150)
    plt.show()

## 4. Summary table

In [ ]:
summary = (
    df.groupby("solver")[["total", "latency", "swap", "priority", "wall_time"]]
    .agg(["mean", "std"])
    .round(3)
)
print("=== Overall averages across all sweep cells ===")
print(summary.to_string())

mean_gap = ratio["gap_pct"].mean()
median_gap = ratio["gap_pct"].median()
print(f"\nGA optimality gap — mean: {mean_gap:.1f}%  median: {median_gap:.1f}%")